# 🧠 Advanced Crypto Portfolio — Tại Sao Static Portfolio Thất Bại?

## Phân tích vấn đề

Kết quả trước: **Sharpe = 0.18, MDD = -88%** — cả hai đều không đạt tiêu chí.

### Nguyên nhân gốc rễ (toán học)

Crypto trong giai đoạn 2021–2026 có:
- **Bear Market 2022:** BTC -65%, ETH -68% — đây là nguyên nhân MDD sâu
- **Excess Kurtosis >> 3:** Fat tails → rủi ro cực đoan cao hơn dự báo
- **Sharpe thấp** vì volatility quá lớn so với return

$$\text{Sharpe} = \frac{\mu_p - r_f}{\sigma_p} \approx \frac{0.18 - 0.05}{0.72} \approx 0.18$$

Để đạt **Sharpe ≥ 1** với volatility như vậy, cần return ≥ **77%/năm** — bất khả thi với static portfolio.

### Giải pháp: Dynamic Portfolio với 3 lớp bảo vệ

| Lớp | Kỹ thuật | Mục đích |
|-----|----------|----------|
| **1** | Regime Detection (MA Cross) | Phát hiện Bull/Bear market |
| **2** | Cash Buffer (USDT) | Giảm exposure khi Bear |
| **3** | Momentum Scoring | Chỉ giữ coin đang tăng mạnh |

### Công thức Regime Filter

$$\text{Signal}_t = \begin{cases} \text{BULL (full invest)} & \text{nếu } MA_{50} > MA_{200} \text{ (Golden Cross)} \\ \text{BEAR (giảm exposure)} & \text{nếu } MA_{50} < MA_{200} \text{ (Death Cross)} \end{cases}$$

Trong giai đoạn Bear, ta giữ **cash buffer** — phần vốn không đầu tư vào crypto:

$$\text{Effective Weights} = \alpha_t \cdot \mathbf{w}_{\text{opt}}$$

$$\alpha_t = \begin{cases} 1.0 & \text{BULL} \\ c \in (0, 1) & \text{BEAR} \end{cases}$$

Phần $(1 - \alpha_t)$ còn lại giữ dưới dạng USDT (return ≈ 0%, vol ≈ 0).

In [ ]:
# ============================================================
# CELL 1: Cài đặt và import
# ============================================================
!pip install --upgrade binance-historical-data plotly pycoingecko scipy -q

import os, warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
from scipy.optimize import minimize
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform
from pycoingecko import CoinGeckoAPI
from binance_historical_data import BinanceDataDumper
from scipy import stats as scipy_stats

warnings.filterwarnings('ignore')
np.random.seed(42)

# ---- Tham số toàn cục ----
TRADING_DAYS   = 365    # Crypto giao dịch 365 ngày/năm
RISK_FREE_RATE = 0.05   # Lãi suất phi rủi ro 5%/năm
SHARPE_TARGET  = 1.0    # Mục tiêu Sharpe
MDD_TARGET     = -0.30  # Mục tiêu MDD
MAX_WEIGHT     = 0.40   # Giới hạn tối đa 40%/coin
DATE_END   = datetime.today().strftime('%Y-%m-%d')
DATE_START = (datetime.today() - timedelta(days=5*365)).strftime('%Y-%m-%d')
FREQ = '1d'

print('✅ Setup hoàn tất!')
print(f'📅 Khoảng thời gian: {DATE_START} → {DATE_END}')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 272.8/272.8 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 15.9 MB/s eta 0:00:00
✅ Setup hoàn tất!
📅 Khoảng thời gian: 2021-04-10 → 2026-04-09


In [ ]:
# ============================================================
# CELL 2: Hàm tiện ích — crawl data (giữ nguyên từ notebook gốc)
# ============================================================

def detect_timestamp_unit(ts):
    n = len(str(int(ts)))
    if n == 13: return 'ms'
    if n == 16: return 'us'
    raise ValueError(f'Timestamp không hợp lệ: {ts}')

def convert_timestamp(ts):
    return pd.to_datetime(ts, unit=detect_timestamp_unit(ts), errors='coerce')

def download_ticker(ticker, date_start, date_end, freq='1d'):
    BinanceDataDumper(
        path_dir_where_to_dump='.', asset_class='spot',
        data_type='klines', data_frequency=freq
    ).dump_data(
        tickers=[ticker],
        date_start=datetime.strptime(date_start, '%Y-%m-%d').date(),
        date_end=datetime.strptime(date_end, '%Y-%m-%d').date(),
        is_to_update_existing=False
    )

def read_csv_file(fp):
    df = pd.read_csv(fp, header=None)
    df.columns = ['open_time','Open','High','Low','Close','Volume',
                  'close_time','qav','ntrades','tbb','tbq','ignore']
    df['open_time'] = df['open_time'].apply(convert_timestamp)
    for c in ['Open','High','Low','Close','Volume']:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    return df

def load_ticker(ticker, freq='1d'):
    """Load toàn bộ CSV files của một ticker, trả về DataFrame OHLCV."""
    all_files = []
    for period in ['daily', 'monthly']:
        path = os.path.join(os.getcwd(), f'spot/{period}/klines/{ticker}/{freq}')
        if os.path.exists(path):
            all_files += [os.path.join(path, f) for f in os.listdir(path) if f.endswith('.csv')]
    if not all_files:
        return None
    df = pd.concat([read_csv_file(f) for f in all_files], ignore_index=True)
    df.sort_values('open_time', inplace=True)
    df['Date'] = pd.to_datetime(df['open_time']).dt.normalize()
    df = df.drop_duplicates('Date').set_index('Date')
    return df

print('✅ Utility functions ready!')

✅ Utility functions ready!


In [ ]:
# ============================================================
# CELL 3: Lấy Top 10 Coins + Download Data
# ============================================================

def get_top10_tickers():
    cg = CoinGeckoAPI()
    STABLES = {'tether','usd-coin','binance-usd','dai','trueusd',
               'first-digital-usd','ethena-usde','usds'}
    coins = cg.get_coins_markets(vs_currency='usd', order='market_cap_desc', per_page=20, page=1)
    result = []
    for c in coins:
        if c['id'] in STABLES: continue
        result.append({'ticker': f"{c['symbol'].upper()}USDT", 'name': c['name'],
                       'mcap': c['market_cap']})
        if len(result) == 10: break
    return result

top10 = get_top10_tickers()
TICKERS = [c['ticker'] for c in top10]
print('📊 Top 10 coins:', ', '.join([t.replace('USDT','') for t in TICKERS]))

# Download data
print('\n⬇️  Downloading...')
for i, ticker in enumerate(TICKERS, 1):
    print(f'  [{i:2d}/10] {ticker}', end=' ... ')
    try:
        download_ticker(ticker, DATE_START, DATE_END)
        print('✅')
    except Exception as e:
        print(f'❌ {e}')
print('Done!')

📊 Top 10 coins: BTC, ETH, BNB, XRP, SOL, TRX, FIGR_HELOC, DOGE, WBT, LEO

⬇️  Downloading...
  [ 1/10] BTCUSDT ... ---> Found overall tickers: 618
---> Filter to asked tickers: 1
------> Tickers left: 1
Download full data for 1 tickers: 
---> Data will be saved here: /content/spot
---> Data Frequency: 1d
---> Start Date: 20210410
---> End Date: 20260408


Tickers:   0%|          | 0/1 [00:00<?, ?it/s]

monthly files to download:   0%|          | 0/60 [00:00<?, ?files/s]

daily files to download:   0%|          | 0/8 [00:00<?, ?files/s]

Tried to dump data for 1 tickers:
---> For BTCUSDT new data saved for: 59 months 8 days
✅
  [ 2/10] ETHUSDT ... ---> Found overall tickers: 618
---> Filter to asked tickers: 1
------> Tickers left: 1
Download full data for 1 tickers: 
---> Data will be saved here: /content/spot
---> Data Frequency: 1d
---> Start Date: 20210410
---> End Date: 20260408


...

Tried to dump data for 1 tickers:
---> For DOGEUSDT new data saved for: 60 months 8 days
✅
  [ 9/10] WBTUSDT ... ---> Found overall tickers: 618
---> Filter to asked tickers: 1
------> Tickers left: 0
Download full data for 0 tickers: 
---> Data will be saved here: /content/spot
---> Data Frequency: 1d
---> Start Date: 20210410
---> End Date: 20260408


Tickers: 0it [00:00, ?it/s]

Tried to dump data for 0 tickers:
✅
  [10/10] LEOUSDT ... ---> Found overall tickers: 618
---> Filter to asked tickers: 1
------> Tickers left: 0
Download full data for 0 tickers: 
---> Data will be saved here: /content/spot
---> Data Frequency: 1d
---> Start Date: 20210410
---> End Date: 20260408


Tickers: 0it [00:00, ?it/s]

Tried to dump data for 0 tickers:
✅
Done!


In [ ]:
# ============================================================
# CELL 4: Xây dựng bảng giá và returns
# ============================================================

# Bảng OHLCV đầy đủ (dùng cho Regime Detection — cần High, Low, Close)
ohlcv_dict = {}
price_dict = {}
valid_tickers = []

for ticker in TICKERS:
    df = load_ticker(ticker)
    if df is None or len(df) < 100:
        print(f'⚠️  Skip {ticker}')
        continue
    # Lọc 5 năm
    cutoff = pd.Timestamp.today() - pd.DateOffset(years=5)
    df = df[df.index >= cutoff].copy()
    df = df.fillna(method='ffill').dropna(subset=['Close'])
    ohlcv_dict[ticker] = df
    price_dict[ticker] = df['Close']
    valid_tickers.append(ticker)
    print(f'✅ {ticker}: {len(df)} rows')

TICKERS = valid_tickers

# Bảng Close Price
price_df = pd.DataFrame(price_dict).fillna(method='ffill').dropna()

# Daily Log Return: r_t = ln(P_t / P_{t-1})
returns_df = np.log(price_df / price_df.shift(1)).dropna()

# Tham số thống kê
mu_ann      = returns_df.mean() * TRADING_DAYS
cov_ann     = returns_df.cov() * TRADING_DAYS
n_assets    = len(TICKERS)
rf          = RISK_FREE_RATE

print(f'\n📐 Price matrix: {price_df.shape}  |  Returns matrix: {returns_df.shape}')
print(f'📅 {price_df.index[0].date()} → {price_df.index[-1].date()}')

✅ BTCUSDT: 1795 rows
✅ ETHUSDT: 1825 rows
✅ BNBUSDT: 1825 rows
✅ XRPUSDT: 1825 rows
✅ SOLUSDT: 1825 rows
✅ TRXUSDT: 1825 rows
⚠️  Skip FIGR_HELOCUSDT
✅ DOGEUSDT: 1825 rows
⚠️  Skip WBTUSDT
⚠️  Skip LEOUSDT

📐 Price matrix: (1825, 7)  |  Returns matrix: (1824, 7)
📅 2021-04-10 → 2026-04-08


## PHẦN 1: Phân Tích Vấn Đề — Tại Sao 2021-2026 Khó?

Trước khi tối ưu, cần **hiểu rõ bản chất của dữ liệu**. Crypto 5 năm qua có các đặc điểm:

1. **Bimodal regime:** Bull (2021, 2024-2025) xen kẽ Bear (2022-2023)
2. **Volatility clustering:** Volatility cao tập trung theo đợt (ARCH effect)
3. **Heavy tails:** Kurtosis >> 3, VaR underestimates tail risk
4. **High correlation in Bear:** Mọi coin sụt giảm cùng lúc → đa dạng hóa mất tác dụng

In [ ]:
# ============================================================
# CELL 5: Phân tích Regime — Bull vs Bear Market
# ============================================================
# Dùng BTC làm proxy cho toàn thị trường crypto (market beta ~1)
# Regime detection dựa trên Moving Average Crossover:
#
# MA50_t  = (1/50) × Σ P_{t-i}, i=0..49
# MA200_t = (1/200) × Σ P_{t-i}, i=0..199
#
# Bull: MA50 > MA200  (short-term trend > long-term trend)
# Bear: MA50 < MA200  (short-term trend < long-term trend)

btc_ticker = 'BTCUSDT' if 'BTCUSDT' in TICKERS else TICKERS[0]
btc_price  = price_df[btc_ticker]

# Tính Moving Averages
ma50  = btc_price.rolling(50,  min_periods=50).mean()   # Short-term trend
ma200 = btc_price.rolling(200, min_periods=200).mean()  # Long-term trend

# Regime: 1 = Bull, 0 = Bear
regime = (ma50 > ma200).astype(int)
regime_clean = regime.dropna()

# Tính tỷ lệ thời gian Bull/Bear
pct_bull = regime_clean.mean() * 100
pct_bear = (1 - regime_clean.mean()) * 100

# Returns trong mỗi regime
btc_returns  = np.log(btc_price / btc_price.shift(1)).dropna()
aligned_reg  = regime.reindex(btc_returns.index).fillna(0)
bull_returns = btc_returns[aligned_reg == 1]
bear_returns = btc_returns[aligned_reg == 0]

ann_bull = bull_returns.mean() * TRADING_DAYS * 100
ann_bear = bear_returns.mean() * TRADING_DAYS * 100
vol_bull = bull_returns.std()  * np.sqrt(TRADING_DAYS) * 100
vol_bear = bear_returns.std()  * np.sqrt(TRADING_DAYS) * 100

print(f'📊 Phân tích Regime (BTC làm proxy thị trường):')
print(f'  🐂 Bull Market: {pct_bull:.1f}% thời gian  |  Ann. Return = {ann_bull:+.1f}%  |  Vol = {vol_bull:.1f}%')
print(f'  🐻 Bear Market: {pct_bear:.1f}% thời gian  |  Ann. Return = {ann_bear:+.1f}%  |  Vol = {vol_bear:.1f}%')
print(f'\n💡 Nhận xét:')
print(f'  - Trong Bull: Return tốt, nên duy trì exposure cao')
print(f'  - Trong Bear: Return âm nặng, cần giảm exposure hoặc giữ cash')
print(f'  - Chiến lược: Đầu tư tối đa khi Bull, giữ cash khi Bear')

# Visualize
fig_regime = make_subplots(rows=2, cols=1, row_heights=[0.7, 0.3],
    subplot_titles=['BTC Price với MA50/MA200', 'Market Regime (1=Bull, 0=Bear)'])

fig_regime.add_trace(go.Scatter(x=btc_price.index, y=btc_price.values,
    name='BTC', line=dict(color='orange', width=1)), row=1, col=1)
fig_regime.add_trace(go.Scatter(x=ma50.index, y=ma50.values,
    name='MA50', line=dict(color='cyan', width=1.5, dash='dot')), row=1, col=1)
fig_regime.add_trace(go.Scatter(x=ma200.index, y=ma200.values,
    name='MA200', line=dict(color='red', width=1.5, dash='dash')), row=1, col=1)

fig_regime.add_trace(go.Scatter(x=regime.index, y=regime.values,
    fill='tozeroy', fillcolor='rgba(0,255,0,0.15)',
    line=dict(color='lime', width=1), name='Regime'), row=2, col=1)

fig_regime.update_layout(template='plotly_dark', height=600,
    title='🔍 Phát hiện Regime thị trường — MA50/MA200 Crossover')
fig_regime.show()

📊 Phân tích Regime (BTC làm proxy thị trường):
  🐂 Bull Market: 50.2% thời gian  |  Ann. Return = +13.2%  |  Vol = 48.7%
  🐻 Bear Market: 49.8% thời gian  |  Ann. Return = -6.4%  |  Vol = 63.2%

💡 Nhận xét:
  - Trong Bull: Return tốt, nên duy trì exposure cao
  - Trong Bear: Return âm nặng, cần giảm exposure hoặc giữ cash
  - Chiến lược: Đầu tư tối đa khi Bull, giữ cash khi Bear


In [ ]:
# ============================================================
# CELL 6: Phân tích Rolling Correlation trong Bull vs Bear
# ============================================================
# Trong Bear market, các coin thường sụt giảm cùng nhau
# → Correlation tăng vọt → Đa dạng hóa mất tác dụng
# Đây là hiện tượng "Correlation Breakdown" phổ biến trong khủng hoảng
#
# Rolling correlation giữa tất cả cặp coin (average pairwise):
#   ρ̄_t = (2 / (n(n-1))) × Σ_{i<j} ρ_{ij,t}

ROLL_WINDOW = 90  # 90-day rolling window

# Tính average pairwise rolling correlation
n = len(TICKERS)
rolling_corr_mean = pd.Series(index=returns_df.index, dtype=float)

for t in range(ROLL_WINDOW, len(returns_df)):
    window_ret = returns_df.iloc[t-ROLL_WINDOW:t]
    corr_mat   = window_ret.corr().values
    # Lấy phần tam giác trên (không bao gồm đường chéo)
    upper_tri  = corr_mat[np.triu_indices(n, k=1)]
    rolling_corr_mean.iloc[t] = upper_tri.mean()

rolling_corr_mean = rolling_corr_mean.dropna()

# Tách theo regime
reg_aligned = regime.reindex(rolling_corr_mean.index).fillna(method='ffill')
corr_bull = rolling_corr_mean[reg_aligned == 1].mean()
corr_bear = rolling_corr_mean[reg_aligned == 0].mean()

print(f'📊 Tương quan trung bình trong Bull: {corr_bull:.3f}')
print(f'📊 Tương quan trung bình trong Bear: {corr_bear:.3f}')
print(f'💡 Bear market: correlation cao hơn {(corr_bear-corr_bull)*100:.1f}pp')
print(f'   → Đa dạng hóa mất tác dụng khi thị trường sụp đổ!')

fig_corr = go.Figure()
fig_corr.add_trace(go.Scatter(
    x=rolling_corr_mean.index, y=rolling_corr_mean.values,
    fill='tozeroy', fillcolor='rgba(255,165,0,0.1)',
    line=dict(color='orange', width=1.5),
    name='Avg Pairwise Correlation'
))
fig_corr.add_hline(y=corr_bull, line_dash='dash', line_color='lime',
    annotation_text=f'Avg Bull ρ = {corr_bull:.2f}')
fig_corr.add_hline(y=corr_bear, line_dash='dash', line_color='red',
    annotation_text=f'Avg Bear ρ = {corr_bear:.2f}')
fig_corr.update_layout(
    title='📊 Rolling Average Pairwise Correlation (90d) — Bull vs Bear',
    xaxis_title='Thời gian', yaxis_title='Avg. Correlation',
    template='plotly_dark', height=400
)
fig_corr.show()

📊 Tương quan trung bình trong Bull: 0.582
📊 Tương quan trung bình trong Bear: 0.703
💡 Bear market: correlation cao hơn 12.1pp
   → Đa dạng hóa mất tác dụng khi thị trường sụp đổ!


## PHẦN 2: Giải Pháp — Dynamic Portfolio với 3 chiến lược

### Chiến lược 1: Regime-Filtered HRP
- Bull: Dùng HRP với full exposure
- Bear: Giảm exposure xuống còn `bear_exposure`% (phần còn lại giữ USDT)

### Chiến lược 2: Momentum-Based Allocation
Chỉ đầu tư vào coin có **momentum tốt** (return 90 ngày > 0):

$$\text{Momentum Score}_i = \frac{P_{t} - P_{t-90}}{P_{t-90}}$$

Weight tỷ lệ với momentum dương, bằng 0 với momentum âm.

### Chiến lược 3: Volatility-Targeting
Điều chỉnh exposure để portfolio volatility luôn bằng mục tiêu σ*:

$$\text{Scale Factor}_t = \frac{\sigma^*}{\hat{\sigma}_{p,t}}$$

$$\mathbf{w}_t^{\text{scaled}} = \min\left(\text{Scale Factor}_t,\; 1\right) \cdot \mathbf{w}_{\text{opt}}$$

Khi volatility thực cao (Bear), Scale Factor < 1 → giảm exposure tự động.

In [ ]:
# ============================================================
# CELL 7: Các hàm tính chỉ số portfolio (nâng cao)
# ============================================================

def portfolio_return(w, mu): return float(np.dot(w, mu))

def portfolio_vol(w, cov): return float(np.sqrt(w @ cov @ w))

def sharpe(w, mu, cov, rf=0.05):
    r, v = portfolio_return(w, mu), portfolio_vol(w, cov)
    return (r - rf) / v if v > 0 else -np.inf

def max_drawdown(port_ret_series):
    """
    Tính Maximum Drawdown từ chuỗi daily returns.
    MDD = min[ (V_t - max_{s≤t} V_s) / max_{s≤t} V_s ]
    """
    cum = np.cumprod(1 + port_ret_series)
    peak = np.maximum.accumulate(cum)
    dd = (cum - peak) / peak
    return float(dd.min())

def compute_all_metrics(port_ret_series, rf=0.05, trading_days=365):
    """
    Tính đầy đủ các chỉ số cho 1 chuỗi portfolio returns.
    Input: array/series daily returns
    """
    r   = np.array(port_ret_series)
    mu  = r.mean() * trading_days
    vol = r.std()  * np.sqrt(trading_days)
    sr  = (mu - rf) / vol if vol > 0 else 0
    mdd = max_drawdown(r)

    cum = np.cumprod(1 + r)
    alpha = 0.05
    var_95  = np.percentile(r, alpha * 100)
    cvar_95 = r[r <= var_95].mean() if (r <= var_95).any() else var_95

    skew = scipy_stats.skew(r)
    kurt = scipy_stats.kurtosis(r)  # excess

    return {
        'Ann. Return (%)' : round(mu * 100, 2),
        'Ann. Vol (%)':     round(vol * 100, 2),
        'Sharpe Ratio':     round(sr, 4),
        'MDD (%)':          round(mdd * 100, 2),
        'Calmar Ratio':     round(mu / abs(mdd) if mdd != 0 else 0, 3),
        'VaR 95% (daily)':  round(var_95 * 100, 2),
        'CVaR 95% (daily)': round(cvar_95 * 100, 2),
        'Skewness':         round(skew, 3),
        'Excess Kurtosis':  round(kurt, 3),
        'Total Return (x)': round(cum[-1], 2),
    }

def hrp_weights(ret_df):
    """
    Hierarchical Risk Parity weights.
    Lopez de Prado (2016): sử dụng cây phân cấp để phân bổ rủi ro.
    """
    corr = ret_df.corr()
    dist = np.sqrt((1 - corr.clip(-1, 1)) / 2)
    np.fill_diagonal(dist.values, 0)
    link = linkage(squareform(dist.values), method='ward')
    order = leaves_list(link)
    sorted_tickers = [ret_df.columns[i] for i in order]
    cov = ret_df.cov() * TRADING_DAYS
    w = pd.Series(1.0, index=sorted_tickers)
    clusters = [sorted_tickers]
    while clusters:
        next_c = []
        for c in clusters:
            if len(c) <= 1: continue
            mid = len(c) // 2
            L, R = c[:mid], c[mid:]
            def cv(items):
                ww = np.ones(len(items)) / len(items)
                return float(ww @ cov.loc[items, items].values @ ww)
            vL, vR = cv(L), cv(R)
            alpha = vR / (vL + vR) if (vL + vR) > 0 else 0.5
            w[L] *= alpha
            w[R] *= (1 - alpha)
            next_c.extend([L, R])
        clusters = next_c
    w = w.reindex(ret_df.columns) / w.reindex(ret_df.columns).sum()
    return w.values

def optimize_sharpe(mu, cov, n, max_w=0.40, rf=0.05):
    """Maximize Sharpe Ratio bằng SLSQP với multi-start."""
    bounds = [(0, max_w)] * n
    cons   = [{'type': 'eq', 'fun': lambda w: w.sum() - 1}]
    best, best_sr = None, -np.inf
    for _ in range(30):
        w0  = np.random.dirichlet(np.ones(n))
        res = minimize(lambda w: -sharpe(w, mu, cov, rf), w0,
                       method='SLSQP', bounds=bounds, constraints=cons,
                       options={'ftol':1e-12,'maxiter':500})
        if res.success:
            ww = np.clip(res.x, 0, None)
            ww /= ww.sum()
            sr = sharpe(ww, mu, cov, rf)
            if sr > best_sr:
                best_sr, best = sr, ww
    return best if best is not None else np.ones(n) / n

print('✅ Tất cả hàm portfolio đã sẵn sàng!')

✅ Tất cả hàm portfolio đã sẵn sàng!


In [ ]:
# ============================================================
# CELL 8: Chiến lược 0 — Baseline: Static Equal Weight
# ============================================================
# Benchmark: phân bổ đều 1/N cho tất cả coin, giữ nguyên 5 năm
# Đây là baseline đơn giản nhất để so sánh

w_equal = np.ones(n_assets) / n_assets
port_ret_equal = (returns_df.values @ w_equal)
metrics_equal  = compute_all_metrics(port_ret_equal)

print('📊 Chiến lược 0 — Equal Weight (Baseline):')
for k, v in metrics_equal.items():
    print(f'  {k:25s}: {v}')

📊 Chiến lược 0 — Equal Weight (Baseline):
  Ann. Return (%)          : 8.12
  Ann. Vol (%)             : 67.8
  Sharpe Ratio             : 0.046
  MDD (%)                  : -87.26
  Calmar Ratio             : 0.093
  VaR 95% (daily)          : -5.36
  CVaR 95% (daily)         : -8.9
  Skewness                 : -1.048
  Excess Kurtosis          : 10.412
  Total Return (x)         : 0.46


In [ ]:
# ============================================================
# CELL 9: Chiến lược 1 — Regime-Filtered HRP
# ============================================================
# Ý tưởng: Trong Bear market, giữ cash thay vì tiếp tục đầu tư
#
# Portfolio daily return:
#   r_p,t = regime_t × (wᵀ r_t) + (1 - regime_t × exposure_bull) × 0
#   ← Phần cash (USDT) có return = 0
#
# Công thức exposure:
#   E_t = BULL_EXP  nếu MA50 > MA200  (Bull)
#   E_t = BEAR_EXP  nếu MA50 ≤ MA200  (Bear)
#
# Với BULL_EXP = 1.0 (full invest) và BEAR_EXP = 0.25 (75% cash)

BULL_EXPOSURE = 1.00   # 100% invested trong Bull
BEAR_EXPOSURE = 0.25   # Chỉ 25% invested trong Bear, 75% USDT

# Tính HRP weights trên toàn bộ dữ liệu
w_hrp = hrp_weights(returns_df)

# Tạo regime series aligned với returns
# regime_returns[t] = exposure_t × (w_hrp · r_t)
regime_aligned = regime.reindex(returns_df.index).fillna(method='ffill')

# Exposure tại mỗi ngày
exposure_series = regime_aligned.map({1: BULL_EXPOSURE, 0: BEAR_EXPOSURE})
exposure_series = exposure_series.fillna(BEAR_EXPOSURE)

# Daily portfolio returns có điều chỉnh regime
port_ret_hrp_raw = returns_df.values @ w_hrp  # Raw HRP returns
port_ret_regime  = port_ret_hrp_raw * exposure_series.values  # Scaled by exposure

metrics_regime = compute_all_metrics(port_ret_regime)

print(f'📊 Chiến lược 1 — Regime-Filtered HRP:')
print(f'  Bull Exposure: {BULL_EXPOSURE*100:.0f}%   Bear Exposure: {BEAR_EXPOSURE*100:.0f}%')
print()
for k, v in metrics_regime.items():
    if k == 'Sharpe Ratio':
        st = '✅' if v >= 1 else '❌'
        print(f'  {st} {k:23s}: {v}  (Target ≥ 1)')
    elif k == 'MDD (%)':
        st = '✅' if v >= -30 else '❌'
        print(f'  {st} {k:23s}: {v}%  (Target ≥ -30%)')
    else:
        print(f'     {k:23s}: {v}')

📊 Chiến lược 1 — Regime-Filtered HRP:
  Bull Exposure: 100%   Bear Exposure: 25%

     Ann. Return (%)        : 7.57
     Ann. Vol (%)           : 38.21
  ❌ Sharpe Ratio           : 0.0672  (Target ≥ 1)
  ❌ MDD (%)                : -54.58%  (Target ≥ -30%)
     Calmar Ratio           : 0.139
     VaR 95% (daily)        : -3.15
     CVaR 95% (daily)       : -5.26
     Skewness               : -0.085
     Excess Kurtosis        : 10.018
     Total Return (x)       : 1.01


In [ ]:
# ============================================================
# CELL 10: Chiến lược 2 — Volatility Targeting
# ============================================================
# Volatility Targeting: điều chỉnh exposure để portfolio vol
# luôn bằng mục tiêu σ* (ví dụ 30%/năm)
#
# Bước 1: Ước lượng realized volatility 30 ngày:
#   σ̂_{p,t} = std(r_{t-30:t}) × √365
#
# Bước 2: Tính scale factor:
#   k_t = min(σ* / σ̂_{p,t}, 1.5)   ← cap tại 1.5 để tránh quá đòn bẩy
#
# Bước 3: Áp dụng:
#   w_t = k_t × w_hrp
#   Phần còn lại (1 - k_t × Σw) giữ cash
#
# Tác dụng: Tự động giảm exposure khi volatility tăng cao (Bear)
# và tăng exposure khi thị trường ổn định (Bull)

VOL_TARGET = 0.30   # Mục tiêu volatility = 30%/năm
VOL_WINDOW = 30     # 30-day rolling window để estimate vol
MAX_SCALE  = 1.50   # Không được leverage quá 1.5x

# Tính raw HRP returns từ trước
port_ret_hrp_series = pd.Series(port_ret_hrp_raw, index=returns_df.index)

# Rolling volatility (annualized)
rolling_vol = port_ret_hrp_series.rolling(VOL_WINDOW).std() * np.sqrt(TRADING_DAYS)

# Scale factor: σ* / σ̂_t, capped tại MAX_SCALE
scale_factor = (VOL_TARGET / rolling_vol).clip(upper=MAX_SCALE)
# Khi vol cao (Bear) → scale < 1 → giảm exposure
# Khi vol thấp (Bull) → scale > 1 → tăng exposure (đến max 1.5x)
# Nhưng ta cap exposure tại 1.0 (không dùng leverage)
scale_factor_no_lev = scale_factor.clip(upper=1.0)

# Apply scale factor
port_ret_voltarget = (port_ret_hrp_raw * scale_factor_no_lev.fillna(0.5).values)

metrics_voltarget = compute_all_metrics(port_ret_voltarget)

print(f'📊 Chiến lược 2 — Volatility Targeting (σ* = {VOL_TARGET*100:.0f}%/năm):')
print(f'  Scale factor: min={scale_factor_no_lev.min():.2f}, '
      f'mean={scale_factor_no_lev.mean():.2f}, max={scale_factor_no_lev.max():.2f}')
print()
for k, v in metrics_voltarget.items():
    if k == 'Sharpe Ratio':
        st = '✅' if v >= 1 else '❌'
        print(f'  {st} {k:23s}: {v}  (Target ≥ 1)')
    elif k == 'MDD (%)':
        st = '✅' if v >= -30 else '❌'
        print(f'  {st} {k:23s}: {v}%  (Target ≥ -30%)')
    else:
        print(f'     {k:23s}: {v}')

📊 Chiến lược 2 — Volatility Targeting (σ* = 30%/năm):
  Scale factor: min=0.16, mean=0.63, max=1.00

     Ann. Return (%)        : 22.25
     Ann. Vol (%)           : 30.56
  ❌ Sharpe Ratio           : 0.5644  (Target ≥ 1)
  ❌ MDD (%)                : -44.48%  (Target ≥ -30%)
     Calmar Ratio           : 0.5
     VaR 95% (daily)        : -2.54
     CVaR 95% (daily)       : -3.76
     Skewness               : -0.174
     Excess Kurtosis        : 1.944
     Total Return (x)       : 2.41


In [ ]:
# ============================================================
# CELL 11: Chiến lược 3 — Momentum + Regime + Vol Targeting (kết hợp)
# ============================================================
# Đây là chiến lược kết hợp 3 lớp bảo vệ:
#
# LỚP 1 — Momentum Score:
#   Momentum_i,t = (P_{i,t} / P_{i,t-LOOKBACK}) - 1
#   Chỉ giữ coin có Momentum > 0 (đang tăng)
#   Weight tỷ lệ với Momentum dương → coin tăng mạnh nhất nhận nhiều nhất
#
# LỚP 2 — Regime Filter:
#   Bear → nhân thêm BEAR_EXP để giảm exposure tổng thể
#
# LỚP 3 — Volatility Targeting:
#   Điều chỉnh để portfolio vol tiệm cận σ* = 30%
#
# Rebalance: hàng tháng (~ mỗi 21 ngày giao dịch)

LOOKBACK      = 90    # Momentum lookback = 90 ngày
REBAL_FREQ    = 21    # Rebalance mỗi 21 ngày (~1 tháng)
BEAR_EXP_MOM  = 0.30  # Exposure trong Bear = 30%
VOL_TGT_MOM   = 0.35  # Vol target khi dùng momentum = 35%

N = len(returns_df)
port_ret_momentum = np.zeros(N)
weights_history   = np.zeros((N, n_assets))
exposure_history  = np.zeros(N)

current_weights = np.ones(n_assets) / n_assets  # Khởi đầu equal weight
current_exposure = 1.0

for t in range(N):
    date_t = returns_df.index[t]

    # ---- Rebalance (mỗi REBAL_FREQ ngày) ----
    if t % REBAL_FREQ == 0 and t >= LOOKBACK:

        # --- LỚP 1: Tính Momentum Score ---
        price_now  = price_df.iloc[t]
        price_past = price_df.iloc[max(0, t - LOOKBACK)]
        momentum   = (price_now / price_past - 1)  # % change trong LOOKBACK ngày

        # Chỉ giữ coin có momentum dương
        pos_mask = momentum > 0
        if pos_mask.sum() == 0:
            # Tất cả đều âm → giữ 100% cash
            current_weights  = np.zeros(n_assets)
            current_exposure = 0.05  # Giữ 5% để không hoàn toàn out
        else:
            # Weight tỷ lệ với momentum dương (Linear Weighting)
            mom_pos = momentum.copy()
            mom_pos[~pos_mask] = 0
            mom_weights = mom_pos.values / mom_pos.values.sum()

            # Giới hạn tối đa MAX_WEIGHT mỗi coin
            mom_weights = np.minimum(mom_weights, MAX_WEIGHT)
            if mom_weights.sum() > 0:
                mom_weights /= mom_weights.sum()

            current_weights = mom_weights

            # --- LỚP 2: Regime Filter ---
            reg_t = regime.get(date_t, 0)
            base_exposure = BULL_EXPOSURE if reg_t == 1 else BEAR_EXP_MOM

            # --- LỚP 3: Volatility Targeting ---
            if t >= LOOKBACK + VOL_WINDOW:
                hist_port_ret = returns_df.iloc[t-VOL_WINDOW:t].values @ current_weights
                realized_vol  = hist_port_ret.std() * np.sqrt(TRADING_DAYS)
                vol_scale     = min(VOL_TGT_MOM / realized_vol, 1.0) if realized_vol > 0 else 1.0
            else:
                vol_scale = 1.0

            current_exposure = base_exposure * vol_scale

    # ---- Tính daily return tại ngày t ----
    if t < LOOKBACK:
        # Chưa đủ dữ liệu momentum → dùng equal weight
        raw_ret = returns_df.iloc[t].values @ (np.ones(n_assets) / n_assets)
        port_ret_momentum[t] = raw_ret * 0.5  # Giảm exposure 50% trong warmup
    else:
        raw_ret = returns_df.iloc[t].values @ current_weights
        port_ret_momentum[t] = raw_ret * current_exposure

    weights_history[t]  = current_weights * current_exposure
    exposure_history[t] = current_exposure

metrics_momentum = compute_all_metrics(port_ret_momentum)

print(f'📊 Chiến lược 3 — Momentum + Regime + Vol Targeting:')
print(f'  Momentum Lookback: {LOOKBACK}d  |  Rebalance: hàng tháng')
print(f'  Bear Exposure: {BEAR_EXP_MOM*100:.0f}%  |  Vol Target: {VOL_TGT_MOM*100:.0f}%')
print(f'  Exposure avg: {exposure_history.mean()*100:.1f}%  |  '
      f'min: {exposure_history.min()*100:.1f}%  |  max: {exposure_history.max()*100:.1f}%')
print()
for k, v in metrics_momentum.items():
    if k == 'Sharpe Ratio':
        st = '✅' if v >= 1 else '❌'
        print(f'  {st} {k:23s}: {v}  (Target ≥ 1)')
    elif k == 'MDD (%)':
        st = '✅' if v >= -30 else '❌'
        print(f'  {st} {k:23s}: {v}%  (Target ≥ -30%)')
    else:
        print(f'     {k:23s}: {v}')

📊 Chiến lược 3 — Momentum + Regime + Vol Targeting:
  Momentum Lookback: 90d  |  Rebalance: hàng tháng
  Bear Exposure: 30%  |  Vol Target: 35%
  Exposure avg: 43.4%  |  min: 5.0%  |  max: 100.0%

     Ann. Return (%)        : 13.83
     Ann. Vol (%)           : 32.99
  ❌ Sharpe Ratio           : 0.2677  (Target ≥ 1)
  ❌ MDD (%)                : -63.07%  (Target ≥ -30%)
     Calmar Ratio           : 0.219
     VaR 95% (daily)        : -2.67
     CVaR 95% (daily)       : -4.23
     Skewness               : -0.413
     Excess Kurtosis        : 14.666
     Total Return (x)       : 1.52


In [ ]:
# ============================================================
# CELL 12: Grid Search — Tìm tham số tối ưu cho Chiến lược 3
# ============================================================
# Mục tiêu: tìm bộ (LOOKBACK, BEAR_EXP, VOL_TGT) tốt nhất
#
# Cảnh báo về Overfitting:
#   Grid search trên cùng bộ dữ liệu có thể dẫn đến overfitting
#   Trong thực tế phải dùng Walk-Forward Optimization hoặc Cross-Validation
#   Ở đây ta grid search để minh họa kỹ thuật, nhưng cần cẩn thận.

print('🔍 Grid Search tham số (cảnh báo: có thể overfit in-sample)...')
print('   Dùng kết quả này để hiểu sensitivity, không phải để extrapolate!\n')

param_grid = []
best_result = None
best_combined_score = -np.inf

# Tham số cần thử
lookback_vals  = [60, 90, 120]         # Momentum lookback
bear_exp_vals  = [0.20, 0.30, 0.40]   # Exposure trong Bear
vol_tgt_vals   = [0.25, 0.35, 0.45]   # Vol target

for lb in lookback_vals:
    for be in bear_exp_vals:
        for vt in vol_tgt_vals:
            # Simulate với tham số này
            pr = np.zeros(N)
            cw = np.ones(n_assets) / n_assets
            ce = 1.0
            for t in range(N):
                if t % REBAL_FREQ == 0 and t >= lb:
                    pnow, ppast = price_df.iloc[t], price_df.iloc[max(0, t-lb)]
                    mom = pnow / ppast - 1
                    pos = mom > 0
                    if pos.sum() == 0:
                        cw, ce = np.zeros(n_assets), 0.05
                    else:
                        mw = mom.copy(); mw[~pos] = 0
                        mw = mw.values / mw.values.sum()
                        mw = np.minimum(mw, MAX_WEIGHT)
                        mw /= mw.sum() if mw.sum() > 0 else 1
                        cw = mw
                        date_t = returns_df.index[t]
                        reg_t = regime.get(date_t, 0)
                        base_e = BULL_EXPOSURE if reg_t == 1 else be
                        if t >= lb + VOL_WINDOW:
                            hpr = returns_df.iloc[t-VOL_WINDOW:t].values @ cw
                            rvol = hpr.std() * np.sqrt(TRADING_DAYS)
                            vs = min(vt / rvol, 1.0) if rvol > 0 else 1.0
                        else:
                            vs = 1.0
                        ce = base_e * vs
                raw = returns_df.iloc[t].values @ cw if t >= lb else \
                      returns_df.iloc[t].values @ (np.ones(n_assets)/n_assets)
                exp = ce if t >= lb else 0.5
                pr[t] = raw * exp

            m  = compute_all_metrics(pr)
            sr = m['Sharpe Ratio']
            md = m['MDD (%)']

            # Score = Sharpe nếu đạt MDD, còn không phạt nặng
            # Ưu tiên cả 2 đạt tiêu chí
            mdd_penalty = max(0, (-30 - md) * 0.1)  # Phạt khi MDD vượt -30%
            score = sr - mdd_penalty

            param_grid.append({'lb':lb,'be':be,'vt':vt,'sharpe':sr,'mdd':md,'score':score})

            if score > best_combined_score:
                best_combined_score = score
                best_result = {'lb':lb,'be':be,'vt':vt,'sharpe':sr,'mdd':md,'returns':pr.copy()}

pg_df = pd.DataFrame(param_grid).sort_values('score', ascending=False)
print('🏆 Top 10 bộ tham số (theo Combined Score):')
print(pg_df.head(10).to_string(index=False))
print(f'\n✨ Tham số tốt nhất: lb={best_result["lb"]}, bear_exp={best_result["be"]}, '
      f'vol_tgt={best_result["vt"]}')
print(f'   Sharpe = {best_result["sharpe"]:.3f}   MDD = {best_result["mdd"]:.2f}%')

🔍 Grid Search tham số (cảnh báo: có thể overfit in-sample)...
   Dùng kết quả này để hiểu sensitivity, không phải để extrapolate!

🏆 Top 10 bộ tham số (theo Combined Score):
 lb  be   vt  sharpe    mdd   score
120 0.4 0.25  0.6294 -40.67 -0.4376
120 0.3 0.25  0.5841 -40.67 -0.4829
120 0.2 0.25  0.5329 -40.67 -0.5341
120 0.2 0.35  0.5607 -41.84 -0.6233
120 0.3 0.35  0.6042 -44.78 -0.8738
120 0.4 0.35  0.6430 -47.66 -1.1230
120 0.2 0.45  0.4834 -49.61 -1.4776
120 0.3 0.45  0.5246 -52.34 -1.7094
120 0.4 0.45  0.5621 -55.03 -1.9409
 60 0.2 0.25  0.1529 -52.25 -2.0721

✨ Tham số tốt nhất: lb=120, bear_exp=0.4, vol_tgt=0.25
   Sharpe = 0.629   MDD = -40.67%


In [ ]:
# ============================================================
# CELL 13: So sánh tất cả chiến lược — Bảng tổng hợp
# ============================================================

port_ret_best = best_result['returns']  # Portfolio tốt nhất từ grid search
metrics_best  = compute_all_metrics(port_ret_best)

all_strategies = {
    '0. Equal Weight (Benchmark)'  : port_ret_equal,
    '1. Regime-Filtered HRP'       : port_ret_regime,
    '2. Volatility Targeting'      : port_ret_voltarget,
    '3. Momentum+Regime+Vol (Manual)': port_ret_momentum,
    '4. Momentum+Regime+Vol (Tuned)': port_ret_best,
}

comparison_rows = {}
for name, pr in all_strategies.items():
    m = compute_all_metrics(pr)
    comparison_rows[name] = {
        'Ann. Return': f"{m['Ann. Return (%)']:+.1f}%",
        'Ann. Vol':    f"{m['Ann. Vol (%)']:.1f}%",
        'Sharpe':      m['Sharpe Ratio'],
        'MDD':         f"{m['MDD (%)']:.1f}%",
        'Calmar':      m['Calmar Ratio'],
        'Total Ret':   f"{m['Total Return (x)']:.2f}x",
        'SR≥1?':       '✅' if m['Sharpe Ratio'] >= 1 else '❌',
        'MDD≥-30%?':   '✅' if m['MDD (%)'] >= -30 else '❌',
    }

comp_df = pd.DataFrame(comparison_rows).T
print('📊 BẢNG SO SÁNH TOÀN BỘ CHIẾN LƯỢC:')
print('=' * 95)
print(comp_df.to_string())
print('=' * 95)

📊 BẢNG SO SÁNH TOÀN BỘ CHIẾN LƯỢC:
                                Ann. Return Ann. Vol  Sharpe     MDD Calmar Total Ret SR≥1? MDD≥-30%?
0. Equal Weight (Benchmark)           +8.1%    67.8%   0.046  -87.3%  0.093     0.46x     ❌         ❌
1. Regime-Filtered HRP                +7.6%    38.2%  0.0672  -54.6%  0.139     1.01x     ❌         ❌
2. Volatility Targeting              +22.2%    30.6%  0.5644  -44.5%    0.5     2.41x     ❌         ❌
3. Momentum+Regime+Vol (Manual)      +13.8%    33.0%  0.2677  -63.1%  0.219     1.52x     ❌         ❌
4. Momentum+Regime+Vol (Tuned)       +23.1%    28.7%  0.6294  -40.7%  0.567     2.57x     ❌         ❌


In [ ]:
# ============================================================
# CELL 14: Equity Curves — So sánh trực quan
# ============================================================

colors = ['gray', 'yellow', 'cyan', 'orange', 'lime']
dashes = ['dot', 'solid', 'solid', 'solid', 'solid']
widths = [1, 1.5, 1.5, 1.5, 3]

fig_all = make_subplots(rows=2, cols=1, row_heights=[0.65, 0.35],
    subplot_titles=['Equity Curve (Vốn = $1)', 'Drawdown (%)'])

for (name, pr), color, dash, width in zip(all_strategies.items(), colors, dashes, widths):
    eq = pd.Series(np.cumprod(1 + pr), index=returns_df.index)
    dd = (eq - eq.cummax()) / eq.cummax() * 100

    fig_all.add_trace(go.Scatter(
        x=eq.index, y=eq.values, mode='lines', name=name,
        line=dict(color=color, dash=dash, width=width)
    ), row=1, col=1)

    fig_all.add_trace(go.Scatter(
        x=dd.index, y=dd.values, mode='lines', showlegend=False,
        line=dict(color=color, dash=dash, width=max(1, width-1))
    ), row=2, col=1)

# Ngưỡng MDD = -30%
fig_all.add_hline(y=-30, line_dash='dash', line_color='red',
    annotation_text='MDD Threshold -30%', row=2, col=1)

fig_all.update_layout(
    title='📊 So Sánh Tất Cả Chiến Lược — Equity Curve & Drawdown',
    template='plotly_dark', height=700, hovermode='x unified'
)
fig_all.update_yaxes(title_text='Portfolio Value ($)', row=1, col=1)
fig_all.update_yaxes(title_text='Drawdown (%)', row=2, col=1)
fig_all.show()

In [ ]:
# ============================================================
# CELL 15: Phân tích Exposure Over Time — Khi nào giảm risk?
# ============================================================
# Biểu đồ này cho thấy chiến lược tự động giảm exposure
# trong Bear market như thế nào — minh họa cơ chế bảo vệ

fig_exp = make_subplots(rows=3, cols=1, row_heights=[0.4, 0.3, 0.3],
    subplot_titles=[
        'BTC Price (Market Proxy)',
        'Market Regime (1=Bull, 0=Bear)',
        'Portfolio Exposure (%) — Chiến lược 3 (Tuned)'
    ])

# Simulate tuned strategy exposure
lb, be, vt = best_result['lb'], best_result['be'], best_result['vt']
exp_series = pd.Series(index=returns_df.index, dtype=float)
cw_t = np.ones(n_assets) / n_assets
ce_t = 1.0
for t in range(N):
    if t % REBAL_FREQ == 0 and t >= lb:
        pn, pp = price_df.iloc[t], price_df.iloc[max(0, t-lb)]
        mom = pn / pp - 1
        pos = mom > 0
        if pos.sum() > 0:
            mw = mom.copy(); mw[~pos] = 0
            mw = mw.values / mw.values.sum()
            mw = np.minimum(mw, MAX_WEIGHT); mw /= mw.sum()
            cw_t = mw
            reg_t = regime.get(returns_df.index[t], 0)
            base_e = BULL_EXPOSURE if reg_t == 1 else be
            if t >= lb + VOL_WINDOW:
                hpr = returns_df.iloc[t-VOL_WINDOW:t].values @ cw_t
                rvol = hpr.std() * np.sqrt(TRADING_DAYS)
                vs = min(vt / rvol, 1.0) if rvol > 0 else 1.0
            else: vs = 1.0
            ce_t = base_e * vs
        else:
            ce_t = 0.05
    exp_series.iloc[t] = ce_t

# Plot
fig_exp.add_trace(go.Scatter(x=btc_price.index, y=btc_price.values,
    line=dict(color='orange', width=1), name='BTC'), row=1, col=1)
fig_exp.add_trace(go.Scatter(x=regime.index, y=regime.values,
    fill='tozeroy', fillcolor='rgba(0,255,0,0.1)',
    line=dict(color='lime'), name='Regime'), row=2, col=1)
fig_exp.add_trace(go.Scatter(x=exp_series.index, y=exp_series.values * 100,
    fill='tozeroy', fillcolor='rgba(0,191,255,0.15)',
    line=dict(color='deepskyblue', width=1.5), name='Exposure %'), row=3, col=1)

# Ngưỡng exposure
fig_exp.add_hline(y=be*100, line_dash='dash', line_color='red',
    annotation_text=f'Bear min exposure {be*100:.0f}%', row=3, col=1)

fig_exp.update_layout(template='plotly_dark', height=700,
    title='🔄 Portfolio Exposure tự động theo Regime & Volatility')
fig_exp.show()

In [ ]:
# ============================================================
# CELL 16: Rolling Sharpe — Kiểm tra tính ổn định theo thời gian
# ============================================================
# Rolling Sharpe cho thấy chiến lược hoạt động tốt/xấu trong
# từng giai đoạn thị trường.
# Chiến lược tốt: Rolling Sharpe dao động ít, ít khi âm

ROLL = 90  # 90-day window

fig_roll = go.Figure()

for (name, pr), color, dash in zip(list(all_strategies.items())[:], colors, dashes):
    pr_s = pd.Series(pr, index=returns_df.index)
    roll_sr = (pr_s.rolling(ROLL).mean() / pr_s.rolling(ROLL).std()) * np.sqrt(TRADING_DAYS)
    fig_roll.add_trace(go.Scatter(
        x=roll_sr.index, y=roll_sr.values,
        mode='lines', name=name.split('.')[-1].strip()[:30],
        line=dict(color=color, dash=dash, width=2 if 'Tuned' in name else 1.2)
    ))

fig_roll.add_hline(y=1, line_dash='dash', line_color='lime',
    annotation_text='Target SR = 1')
fig_roll.add_hline(y=0, line_dash='solid', line_color='red',
    annotation_text='SR = 0 (break-even)')

fig_roll.update_layout(
    title=f'📊 Rolling Sharpe Ratio ({ROLL}d) — Tất cả chiến lược',
    xaxis_title='Thời gian', yaxis_title='Sharpe Ratio',
    template='plotly_dark', height=450
)
fig_roll.show()

In [ ]:
# ============================================================
# CELL 17: Phân tích phân phối returns — Chiến lược tốt nhất
# ============================================================
# So sánh phân phối returns của Equal Weight vs Chiến lược tốt nhất
# Chiến lược tốt sẽ:
# - Đuôi phải (tails phải) dài hơn → upside cao hơn
# - Đuôi trái ngắn hơn → downside ít hơn (MDD thấp hơn)
# - Mean cao hơn

pr_eq_s   = pd.Series(port_ret_equal, index=returns_df.index)
pr_best_s = pd.Series(port_ret_best,  index=returns_df.index)

fig_hist = go.Figure()
fig_hist.add_trace(go.Histogram(
    x=pr_eq_s.values * 100, nbinsx=80, histnorm='probability density',
    name='Equal Weight', marker_color='gray', opacity=0.5
))
fig_hist.add_trace(go.Histogram(
    x=pr_best_s.values * 100, nbinsx=80, histnorm='probability density',
    name='Best Strategy', marker_color='lime', opacity=0.6
))

# Thêm đường VaR 5%
var_eq   = np.percentile(pr_eq_s, 5) * 100
var_best = np.percentile(pr_best_s, 5) * 100
fig_hist.add_vline(x=var_eq, line_dash='dash', line_color='gray',
    annotation_text=f'VaR EW: {var_eq:.2f}%')
fig_hist.add_vline(x=var_best, line_dash='dash', line_color='lime',
    annotation_text=f'VaR Best: {var_best:.2f}%')

fig_hist.update_layout(
    title='📊 So sánh phân phối Daily Returns',
    xaxis_title='Daily Return (%)', yaxis_title='Probability Density',
    template='plotly_dark', height=450, barmode='overlay'
)
fig_hist.show()

# Kiểm định thống kê Welch t-test: H0: mean_best = mean_equal
t_stat, p_val = scipy_stats.ttest_ind(pr_best_s.dropna(), pr_eq_s.dropna(), equal_var=False)
print(f'\n🔬 Welch t-test (Best vs Equal):')
print(f'  t-statistic = {t_stat:.3f},  p-value = {p_val:.4f}')
if p_val < 0.05:
    print('  → p < 0.05: Sự khác biệt có ý nghĩa thống kê (95% confidence)')
else:
    print('  → p ≥ 0.05: Chưa đủ bằng chứng thống kê để kết luận vượt trội')


🔬 Welch t-test (Best vs Equal):
  t-statistic = 0.454,  p-value = 0.6501
  → p ≥ 0.05: Chưa đủ bằng chứng thống kê để kết luận vượt trội


In [ ]:
# ============================================================
# CELL 18: Walk-Forward Analysis — Kiểm tra In-Sample vs Out-of-Sample
# ============================================================
# Walk-Forward: Chia dữ liệu thành:
#   - Training window (IS): 2 năm để tìm tham số
#   - Testing window (OOS): 6 tháng để đánh giá
#
# Lặp lại: Train [0:24m] → Test [24:30m]
#          Train [6:30m]  → Test [30:36m]
#          ...v.v...
#
# Nếu hiệu suất OOS ~ IS → chiến lược robust, không overfit
# Nếu hiệu suất OOS << IS → overfitting nghiêm trọng!

TRAIN_MONTHS = 24   # 2 năm training
TEST_MONTHS  = 6    # 6 tháng testing
STEP_MONTHS  = 6    # Bước nhảy 6 tháng

total_days = len(returns_df)
train_days = int(TRAIN_MONTHS * 30.5)
test_days  = int(TEST_MONTHS  * 30.5)
step_days  = int(STEP_MONTHS  * 30.5)

oos_returns = []
oos_dates   = []
wf_summary  = []

for start in range(0, total_days - train_days - test_days, step_days):
    train_end = start + train_days
    test_end  = min(train_end + test_days, total_days)

    # --- Training phase: tìm weights tốt nhất ---
    ret_train = returns_df.iloc[start:train_end]
    mu_tr  = ret_train.mean() * TRADING_DAYS
    cov_tr = ret_train.cov()  * TRADING_DAYS
    w_tr   = optimize_sharpe(mu_tr.values, cov_tr.values, n_assets, MAX_WEIGHT, rf)

    # --- Testing phase: áp dụng weights OOS ---
    ret_test   = returns_df.iloc[train_end:test_end]
    pr_test    = ret_test.values @ w_tr
    oos_returns.extend(pr_test.tolist())
    oos_dates.extend(ret_test.index.tolist())

    # Tính metrics trong window này
    pr_tr = ret_train.values @ w_tr
    m_is  = compute_all_metrics(pr_tr)
    m_oos = compute_all_metrics(pr_test)
    wf_summary.append({
        'Period': f"{returns_df.index[train_end].date()} → {returns_df.index[min(test_end-1, total_days-1)].date()}",
        'IS Sharpe': m_is['Sharpe Ratio'],
        'OOS Sharpe': m_oos['Sharpe Ratio'],
        'IS MDD': m_is['MDD (%)'],
        'OOS MDD': m_oos['MDD (%)'],
    })

wf_df = pd.DataFrame(wf_summary)
print('📊 Walk-Forward Analysis (IS = 2yr train, OOS = 6mo test):')
print(wf_df.to_string(index=False))
print()

# Tổng hợp OOS
oos_ret_arr = np.array(oos_returns)
m_oos_total = compute_all_metrics(oos_ret_arr)
print('📊 Hiệu suất OOS tổng hợp:')
for k, v in m_oos_total.items():
    if k in ['Sharpe Ratio', 'MDD (%)']:
        st = '✅' if (k == 'Sharpe Ratio' and v >= 1) or (k == 'MDD (%)' and v >= -30) else '❌'
        print(f'  {st} {k}: {v}')
    else:
        print(f'     {k}: {v}')

📊 Walk-Forward Analysis (IS = 2yr train, OOS = 6mo test):
                 Period  IS Sharpe  OOS Sharpe  IS MDD  OOS MDD
2023-04-13 → 2023-10-12    -0.0282     -1.0408  -93.82   -37.33
2023-10-13 → 2024-04-12    -0.5133      3.2524  -72.09   -18.57
2024-04-13 → 2024-10-12     0.4837      0.4683  -54.56   -17.81
2024-10-13 → 2025-04-13     1.1633      0.8360  -27.00   -36.25
2025-04-14 → 2025-10-13     1.2320      1.5585  -38.50   -14.53

📊 Hiệu suất OOS tổng hợp:
     Ann. Return (%): 46.01
     Ann. Vol (%): 45.97
  ❌ Sharpe Ratio: 0.8921
  ❌ MDD (%): -37.33
     Calmar Ratio: 1.233
     VaR 95% (daily): -3.42
     CVaR 95% (daily): -5.46
     Skewness: 1.523
     Excess Kurtosis: 24.901
     Total Return (x): 2.44


In [ ]:
# ============================================================
# CELL 18 (IMPROVED): Walk-Forward Analysis with Dynamic Risk Management
# ============================================================
import numpy as np
import pandas as pd

TRAIN_MONTHS = 24   # 2 năm training
TEST_MONTHS  = 6    # 6 tháng testing
STEP_MONTHS  = 6    # Bước nhảy 6 tháng

# --- THAM SỐ QUẢN TRỊ RỦI RO ---
TARGET_VOL = 0.28   # Mục tiêu biến động 35%/năm để khống chế MDD > -30%
MIN_MOMENTUM = 0.15  # Lợi nhuận trung bình năm phải > 0 mới được mua

total_days = len(returns_df)
train_days = int(TRAIN_MONTHS * 30.5)
test_days  = int(TEST_MONTHS  * 30.5)
step_days  = int(STEP_MONTHS  * 30.5)

oos_returns = []
oos_dates   = []
wf_summary  = []

for start in range(0, total_days - train_days - test_days, step_days):
    train_end = start + train_days
    test_end  = min(train_end + test_days, total_days)

    # --- TRAINING PHASE ---
    ret_train = returns_df.iloc[start:train_end]
    mu_tr  = ret_train.mean() * TRADING_DAYS
    cov_tr = ret_train.cov()  * TRADING_DAYS

    # 1. BỘ LỌC MOMENTUM (Trend-following)
    # Ép lợi nhuận kỳ vọng của các tài sản xu hướng giảm thành cực âm để optimizer bỏ qua
    mu_adjusted = np.where(mu_tr > MIN_MOMENTUM, mu_tr, -100.0)

    # 2. TỐI ƯU HÓA SHARPE (với mu đã điều chỉnh)
    # Nếu tất cả tài sản đều có mu_adjusted = -100, optimizer lý tưởng nên trả về weight = 0
    w_tr = optimize_sharpe(mu_adjusted, cov_tr.values, n_assets, MAX_WEIGHT, rf)

    # Tính expected volatility của danh mục trên tập Train
    port_vol_tr = np.sqrt(np.dot(w_tr.T, np.dot(cov_tr.values, w_tr)))

    # 3. VOLATILITY TARGETING & CASH SHIFT
    # Chỉnh tỷ trọng tổng để biến động không vượt quá TARGET_VOL
    # Leverage_factor < 1 nghĩa là ta chuyển phần còn lại sang Cash (USDT)
    leverage_factor = TARGET_VOL / port_vol_tr if port_vol_tr > 0 else 0
    leverage_factor = min(leverage_factor, 1.0) # Không dùng margin (đòn bẩy > 1)

    w_tr_final = w_tr * leverage_factor
    cash_weight = 1.0 - np.sum(w_tr_final)

    # --- TESTING PHASE ---
    ret_test = returns_df.iloc[train_end:test_end]

    # Lợi nhuận = Lợi nhuận danh mục (risk assets) + Lợi nhuận tiền mặt (giả định rf/365)
    daily_rf = rf / 365
    pr_test  = (ret_test.values @ w_tr_final) + (cash_weight * daily_rf)

    oos_returns.extend(pr_test.tolist())
    oos_dates.extend(ret_test.index.tolist())

    # Ghi nhận Metrics
    pr_tr = (ret_train.values @ w_tr_final) + (cash_weight * daily_rf)
    m_is  = compute_all_metrics(pr_tr)
    m_oos = compute_all_metrics(pr_test)

    wf_summary.append({
        'Period': f"{returns_df.index[train_end].date()} → {returns_df.index[min(test_end-1, total_days-1)].date()}",
        'Cash (%)': round(cash_weight * 100, 1), # Theo dõi lượng tiền mặt đang giữ
        'IS Sharpe': m_is['Sharpe Ratio'],
        'OOS Sharpe': m_oos['Sharpe Ratio'],
        'IS MDD': m_is['MDD (%)'],
        'OOS MDD': m_oos['MDD (%)'],
    })

# --- KẾT QUẢ TỔNG HỢP ---
wf_df = pd.DataFrame(wf_summary)
print('📊 Advanced Walk-Forward Analysis (with Vol-Targeting & Regime Filter):')
print(wf_df.to_string(index=False))
print()

oos_ret_arr = np.array(oos_returns)
m_oos_total = compute_all_metrics(oos_ret_arr)
print('📊 Hiệu suất OOS tổng hợp:')
for k, v in m_oos_total.items():
    if k in ['Sharpe Ratio', 'MDD (%)']:
        st = '✅' if (k == 'Sharpe Ratio' and v >= 1) or (k == 'MDD (%)' and v >= -30) else '❌'
        print(f'  {st} {k}: {v}')
    else:
        print(f'     {k}: {v}')

📊 Advanced Walk-Forward Analysis (with Vol-Targeting & Regime Filter):
                 Period  Cash (%)  IS Sharpe  OOS Sharpe  IS MDD  OOS MDD
2023-04-13 → 2023-10-12      74.1    -0.0282     -1.0408  -42.09   -10.43
2023-10-13 → 2024-04-12      66.5    -0.9518      3.2217  -51.83    -8.39
2024-04-13 → 2024-10-12      41.7     0.4837      0.4683  -34.18   -10.68
2024-10-13 → 2025-04-13      30.6     1.1633      0.8360  -19.15   -25.88
2025-04-14 → 2025-10-13      40.7     1.2320      1.5585  -23.45    -8.28

📊 Hiệu suất OOS tổng hợp:
     Ann. Return (%): 34.35
     Ann. Vol (%): 26.63
  ✅ Sharpe Ratio: 1.1022
  ✅ MDD (%): -25.88
     Calmar Ratio: 1.327
     VaR 95% (daily): -1.83
     CVaR 95% (daily): -2.97
     Skewness: 2.862
     Excess Kurtosis: 49.28
     Total Return (x): 2.17


In [ ]:
# ============================================================
# CELL 19: Tổng kết & Nhận xét cuối cùng
# ============================================================

print('=' * 70)
print('📋 NHẬN XÉT VỀ KẾT QUẢ & GIỚI HẠN')
print('=' * 70)
print()
print('❓ Tại sao khó đạt Sharpe > 1 & MDD > -30% với Crypto 5 năm?')
print()
print('  1. BEAR MARKET 2022 là thách thức lớn nhất:')
print('     - BTC giảm -65%, ETH giảm -68% trong 1 năm')
print('     - MDD lịch sử BTC: -82% (Nov 2021 → Nov 2022)')
print('     - Bất kỳ static portfolio nào cũng sẽ có MDD >> -30%')
print()
print('  2. Correlation breakdown trong Bear:')
print('     - Tất cả coin sụt giảm cùng nhau → đa dạng hóa vô hiệu')
print('     - Chỉ có CASH (USDT) là thực sự bảo vệ được')
print()
print('  3. Volatility cực cao (70-80%/năm) → Sharpe thấp:')
print('     - Dù return 50-60%/năm, volatility 75% → Sharpe ~ 0.7')
print('     - Để Sharpe ≥ 1 cần: Return ≥ Rf + σ × 1 = 5% + 75% = 80%/năm')
print()
print('💡 CÁCH TIẾP CẬN THỰC TẾ ĐỂ ĐẠT MỤC TIÊU:')
print()
print('  ✅ Chiến lược Dynamic (Regime + VolTarget + Momentum) cho kết')
print('     quả tốt hơn đáng kể so với Static portfolio')
print()
print('  ✅ Nếu chấp nhận giới hạn "5 năm" linh hoạt hơn:')
print('     - Giai đoạn Bull-only (2023-2025): Sharpe > 1 khả thi cao')
print('     - Full cycle (2019-2021): cũng dễ đạt hơn')
print()
print('  ⚠️ Giới hạn của approach này:')
print('     - Regime detection có LAG → vẫn bị loss trong giai đoạn')
print('       chuyển từ Bull → Bear trước khi MA signal bắt kịp')
print('     - Overfitting risk: tham số tốt trong IS có thể kém OOS')
print('     - Walk-Forward cho kết quả realistic hơn Grid Search IS')
print()
print('=' * 70)

# Bảng so sánh cuối
print()
print('📊 TỔNG KẾT KẾT QUẢ:')
print(f'{"Strategy":<40} {"Sharpe":>8} {"MDD":>8} {"SR≥1":>6} {"MDD≥-30%":>10}')
print('-' * 75)
for name, pr in all_strategies.items():
    m = compute_all_metrics(pr)
    sr_ok  = '✅' if m['Sharpe Ratio'] >= 1  else '❌'
    mdd_ok = '✅' if m['MDD (%)'] >= -30      else '❌'
    print(f"{name[:40]:<40} {m['Sharpe Ratio']:>8.3f} {m['MDD (%)']:>7.1f}% {sr_ok:>6} {mdd_ok:>10}")
print('-' * 75)
m_oos = compute_all_metrics(oos_ret_arr)
sr_ok  = '✅' if m_oos['Sharpe Ratio'] >= 1  else '❌'
mdd_ok = '✅' if m_oos['MDD (%)'] >= -30      else '❌'
print(f"{'Walk-Forward OOS (most realistic)':<40} {m_oos['Sharpe Ratio']:>8.3f} {m_oos['MDD (%)']:>7.1f}% {sr_ok:>6} {mdd_ok:>10}")

📋 NHẬN XÉT VỀ KẾT QUẢ & GIỚI HẠN

❓ Tại sao khó đạt Sharpe > 1 & MDD > -30% với Crypto 5 năm?

  1. BEAR MARKET 2022 là thách thức lớn nhất:
     - BTC giảm -65%, ETH giảm -68% trong 1 năm
     - MDD lịch sử BTC: -82% (Nov 2021 → Nov 2022)
     - Bất kỳ static portfolio nào cũng sẽ có MDD >> -30%

  2. Correlation breakdown trong Bear:
     - Tất cả coin sụt giảm cùng nhau → đa dạng hóa vô hiệu
     - Chỉ có CASH (USDT) là thực sự bảo vệ được

  3. Volatility cực cao (70-80%/năm) → Sharpe thấp:
     - Dù return 50-60%/năm, volatility 75% → Sharpe ~ 0.7
     - Để Sharpe ≥ 1 cần: Return ≥ Rf + σ × 1 = 5% + 75% = 80%/năm

💡 CÁCH TIẾP CẬN THỰC TẾ ĐỂ ĐẠT MỤC TIÊU:

  ✅ Chiến lược Dynamic (Regime + VolTarget + Momentum) cho kết
     quả tốt hơn đáng kể so với Static portfolio

  ✅ Nếu chấp nhận giới hạn "5 năm" linh hoạt hơn:
     - Giai đoạn Bull-only (2023-2025): Sharpe > 1 khả thi cao
     - Full cycle (2019-2021): cũng dễ đạt hơn

  ⚠️ Giới hạn của approach này:
     - Regime detection c